# STARRT.ai: second reviewer round (GPU)

**Settings:** Accelerator = **GPU T4 x2**, Internet = **On**. *Add Input* → the `starrt-gpu2` dataset.

Run cell 1 and the smoke test first. Then **Save Version → Save & Run All (Commit)**. Total about 4 hours. Results → `starrt_gpu2_results.zip`.

In [ ]:
import glob, os, subprocess
hits = glob.glob('/kaggle/input/**/run_distilbert.py', recursive=True)
assert hits, 'Dataset not attached: Add Input -> starrt-gpu2'
CODE = os.path.dirname(hits[0]); print('code folder:', CODE)
import torch; print('GPUs:', torch.cuda.device_count(), [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])

def run(script, args=''):
    cmd = f'cd /kaggle/working && python -u {CODE}/{script} {args}'
    print('>>', cmd, flush=True); r = subprocess.run(cmd, shell=True)
    if r.returncode != 0: print('!! FAILED:', script, args, '(continuing with the next step)')

## 0. Smoke test (3 min)

In [ ]:
run('run_distilbert.py', '--datasets djinni --folds 0 --epochs 1 --limit 300 --skip_forms --tag smoke --eval_variants djinni_typo')
run('run_llm_zeroshot.py', '--datasets faruque --limit 2 --prompt_variant plain --tag smoke_llm')
print('smoke test done: check above that both printed a [smoke ...] result line')

## A. Stack Overflow with two more DistilBERT seeds (~1 h)

In [ ]:
for seed in [13, 7]:
    run('run_distilbert.py', f'--datasets stackoverflow --seed {seed} --tag distilbert_seed{seed} --skip_forms')

## B. Robustness: DistilBERT tested on typos, shuffled word order and dropped words (~40 min)

In [ ]:
run('run_distilbert.py', '--datasets djinni --tag distilbert_robust --skip_forms --eval_variants djinni_typo,djinni_shuffle,djinni_drop')

## C. Domain encoder: JobBERT (BERT further trained on job postings) (~1.3 h)

In [ ]:
run('run_distilbert.py', '--model jjzha/jobbert-base-cased --datasets djinni,faruque --tag jobbert --skip_forms')

## D. Zero-shot LLM: two other prompts (3B) and a larger model (7B on both GPUs) (~1 h)

In [ ]:
for v in ['nodef', 'plain']:
    run('run_llm_zeroshot.py', f'--datasets djinni,faruque --prompt_variant {v} --tag llm_qwen2.5-3b_{v}')
run('run_llm_zeroshot.py', '--model Qwen/Qwen2.5-7B-Instruct --device_map auto --datasets djinni,faruque --tag llm_qwen2.5-7b_zeroshot')

## Zip results
Download `starrt_gpu2_results.zip` from the **Output** tab and send it in the chat.

In [ ]:
import shutil
for f in glob.glob('/kaggle/working/results/smoke*'): os.remove(f)
shutil.make_archive('/kaggle/working/starrt_gpu2_results', 'zip', '/kaggle/working/results')
print(sorted(os.listdir('/kaggle/working/results')))